In [13]:
import pandas as pd

# Load the cleaned master dataset from Day 3A
race_driver = pd.read_csv("../data/processed/clean_race_driver.csv")

In [14]:
# Load raw drivers and constructors
drivers = pd.read_csv("../data/raw/drivers.csv")
constructors = pd.read_csv("../data/raw/constructors.csv")

# Create full driver name
drivers_clean = drivers.loc[:, ["driverId", "forename", "surname"]].copy()
drivers_clean["driverName"] = drivers_clean["forename"] + " " + drivers_clean["surname"]

# Merge driver names into race_driver
race_driver = race_driver.merge(
    drivers_clean[["driverId", "driverName"]],
    on="driverId",
    how="left"
)

# Clean constructor names
constructors_clean = constructors.loc[:, ["constructorId", "name"]].rename(
    columns={"name": "constructorName"}
)

# Merge constructor names into race_driver
race_driver = race_driver.merge(
    constructors_clean,
    on="constructorId",
    how="left"
)

In [15]:
# 3A — Finish position
race_driver["finishPosition"] = race_driver["positionOrder"]

In [16]:
# 3B — Average lap time (convert ms to seconds)
race_driver["avgLapTime_s"] = race_driver["avgLapTime_ms"] / 1000

In [17]:
# 3C — Constructor points (aggregate points per constructor per race)
constructor_points = (
    race_driver.groupby(["raceId", "constructorId"])["points"]
    .sum()
    .reset_index()
)
constructor_points.rename(columns={"points": "constructorPoints"}, inplace=True)

# Merge constructor points back into race_driver
race_driver = race_driver.merge(
    constructor_points, on=["raceId", "constructorId"], how="left"
)

In [18]:
print("\nSanity check — first 5 rows:")
print(race_driver[[
    "raceId", "driverId", "driverName", "constructorId", "constructorName",
    "finishPosition", "avgLapTime_s", "constructorPoints"
]].head())

print("\nMissing values per column:\n", race_driver.isna().sum())


Sanity check — first 5 rows:
   raceId  driverId         driverName  constructorId constructorName  \
0      18         1     Lewis Hamilton              1         McLaren   
1      18         2      Nick Heidfeld              2      BMW Sauber   
2      18         3       Nico Rosberg              3        Williams   
3      18         4    Fernando Alonso              4         Renault   
4      18         5  Heikki Kovalainen              1         McLaren   

   finishPosition  avgLapTime_s  constructorPoints  
0               1     98.114069               14.0  
1               2     98.208517                8.0  
2               3     98.254810                9.0  
3               4     98.410293                5.0  
4               5     98.424655               14.0  

Missing values per column:
 raceId                    0
driverId                  0
constructorId             0
grid                      0
positionOrder             0
points                    0
statusId        

In [19]:
race_driver.to_csv("../data/processed/race_driver_labels.csv", index=False)
print("\nLabeled dataset saved at ../data/processed/race_driver_labels.csv")


Labeled dataset saved at ../data/processed/race_driver_labels.csv
